<a href="https://colab.research.google.com/github/workatustadarshajay/Docs_Code_Testing/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install \
    pymupdf \
    presidio-analyzer \
    presidio-anonymizer \
    spacy \
    pytesseract \
    pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 26.4.0 requires cryptography<51,>=49.0.0, but you have cryptography 48.0.1 which is incompatible.


In [2]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
!apt-get -qq update
!apt-get -qq install tesseract-ocr

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
import os
import re
import json
import math
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import fitz  # PyMuPDF
import pytesseract

from PIL import Image

from google.colab import files

from presidio_analyzer import (
    AnalyzerEngine,
    RecognizerRegistry,
    PatternRecognizer,
    Pattern,
)

In [5]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("pdf-masking")

PII_ENTITIES = [
    "PERSON",
    "EMAIL_ADDRESS",
    "PHONE_NUMBER",
    "CREDIT_CARD",
    "US_SSN",
    "IBAN_CODE",
    "IP_ADDRESS",
    "LOCATION",
    "DATE_TIME",
    "URL",
]

In [7]:
registry = RecognizerRegistry()

registry.load_predefined_recognizers()

analyzer = AnalyzerEngine(
    registry=registry
)

print("Presidio initialized successfully.")

Presidio initialized successfully.


In [8]:
member_id_pattern = Pattern(
    name="member_id",
    regex=r"\b[A-Z]{2,5}[- ]?\d{6,12}\b",
    score=0.70,
)

member_id_recognizer = PatternRecognizer(
    supported_entity="MEMBER_ID",
    patterns=[member_id_pattern],
)


policy_pattern = Pattern(
    name="policy_number",
    regex=r"\b(?:POL|POLICY)[- ]?[A-Z0-9]{5,15}\b",
    score=0.75,
)

policy_recognizer = PatternRecognizer(
    supported_entity="POLICY_NUMBER",
    patterns=[policy_pattern],
)


mrn_pattern = Pattern(
    name="medical_record_number",
    regex=r"\b(?:MRN|MR)[- ]?\d{5,15}\b",
    score=0.80,
)

mrn_recognizer = PatternRecognizer(
    supported_entity="MEDICAL_RECORD_NUMBER",
    patterns=[mrn_pattern],
)


authorization_pattern = Pattern(
    name="authorization_number",
    regex=r"\b(?:AUTH|AUTHORIZATION)[- ]?[A-Z0-9]{5,20}\b",
    score=0.80,
)

authorization_recognizer = PatternRecognizer(
    supported_entity="AUTHORIZATION_NUMBER",
    patterns=[authorization_pattern],
)


registry.add_recognizer(member_id_recognizer)
registry.add_recognizer(policy_recognizer)
registry.add_recognizer(mrn_recognizer)
registry.add_recognizer(authorization_recognizer)

analyzer = AnalyzerEngine(
    registry=registry
)

print("Custom healthcare recognizers added.")

Custom healthcare recognizers added.


In [9]:
uploaded = files.upload()

pdf_name = next(iter(uploaded))

print(f"Uploaded: {pdf_name}")

Saving dummy1.pdf to dummy1.pdf
Uploaded: dummy1.pdf


In [10]:
def detect_pdf_type(pdf_path: str):
    doc = fitz.open(pdf_path)

    total_chars = 0
    page_info = []

    for page_number, page in enumerate(doc, start=1):

        text = page.get_text("text")

        chars = len(text.strip())

        total_chars += chars

        page_info.append({
            "page": page_number,
            "characters": chars,
        })

    doc.close()

    if total_chars > 50:
        pdf_type = "TEXT"

    else:
        pdf_type = "SCANNED"

    return pdf_type, total_chars, page_info

In [12]:
pdf_type, total_chars, page_info = detect_pdf_type(
    pdf_name
)

print("PDF type:", pdf_type)
print("Extracted characters:", total_chars)

print("\nPage information:")

for page in page_info:
    print(page)

PDF type: TEXT
Extracted characters: 3377

Page information:
{'page': 1, 'characters': 2295}
{'page': 2, 'characters': 514}
{'page': 3, 'characters': 568}


In [13]:
@dataclass
class PDFWord:
    page_number: int
    text: str
    x0: float
    y0: float
    x1: float
    y1: float
    start: int
    end: int

In [14]:
def extract_pdf_words(pdf_path: str):

    doc = fitz.open(pdf_path)

    words = []

    for page_number, page in enumerate(doc, start=1):

        page_words = page.get_text("words")

        # Each word:
        #
        # x0, y0, x1, y1, word,
        # block_no, line_no, word_no

        page_text_offset = 0

        for item in page_words:

            x0, y0, x1, y1, text = item[:5]

            if not text.strip():
                continue

            start = page_text_offset

            end = start + len(text)

            words.append(
                PDFWord(
                    page_number=page_number,
                    text=text,
                    x0=x0,
                    y0=y0,
                    x1=x1,
                    y1=y1,
                    start=start,
                    end=end,
                )
            )

            page_text_offset = end + 1

    doc.close()

    return words

In [15]:
@dataclass
class PageText:

    page_number: int
    text: str
    words: List[PDFWord]

In [16]:
def extract_pages(pdf_path: str):

    doc = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(doc, start=1):

        words_raw = page.get_text("words")

        words = []

        reconstructed = []

        current_offset = 0

        for item in words_raw:

            x0, y0, x1, y1, word = item[:5]

            if not word.strip():
                continue

            if reconstructed:
                reconstructed.append(" ")
                current_offset += 1

            start = current_offset

            reconstructed.append(word)

            current_offset += len(word)

            end = current_offset

            words.append(
                PDFWord(
                    page_number=page_number,
                    text=word,
                    x0=x0,
                    y0=y0,
                    x1=x1,
                    y1=y1,
                    start=start,
                    end=end,
                )
            )

        text = "".join(reconstructed)

        pages.append(
            PageText(
                page_number=page_number,
                text=text,
                words=words,
            )
        )

    doc.close()

    return pages

In [17]:
def detect_pii(text: str):

    results = analyzer.analyze(
        text=text,
        language="en",
        score_threshold=0.35,
    )

    return results

In [18]:
sample = """
Patient John Smith
Email john.smith@example.com
Phone 9876543210
"""

results = detect_pii(sample)

for result in results:

    print(
        result.entity_type,
        result.start,
        result.end,
        result.score,
        repr(
            sample[
                result.start:result.end
            ]
        )
    )

EMAIL_ADDRESS 26 48 1.0 'john.smith@example.com'
UK_NHS 55 65 1.0 '9876543210'
PERSON 9 19 0.85 'John Smith'
PHONE_NUMBER 55 65 0.75 '9876543210'
MEMBER_ID 49 65 0.7 'Phone 9876543210'
URL 26 33 0.5 'john.sm'
URL 37 48 0.5 'example.com'


In [19]:
def overlaps(
    a_start,
    a_end,
    b_start,
    b_end,
):

    return (
        max(a_start, b_start)
        <
        min(a_end, b_end)
    )

In [20]:
def entity_to_words(
    entity,
    words: List[PDFWord],
):

    matched = []

    for word in words:

        if overlaps(
            entity.start,
            entity.end,
            word.start,
            word.end,
        ):

            matched.append(word)

    return matched

In [26]:
def words_to_rect(words):

    if not words:
        return None

    x0 = min(w.x0 for w in words)
    y0 = min(w.y0 for w in words)

    x1 = max(w.x1 for w in words)
    y1 = max(w.y1 for w in words)

    # Small padding
    padding_x = 1.5
    padding_y = 1.0

    return fitz.Rect(
        x0 - padding_x,
        y0 - padding_y,
        x1 + padding_x,
        y1 + padding_y,
    )

In [21]:
def entity_to_words(
    entity,
    words: List[PDFWord],
):

    matched = []

    for word in words:

        if overlaps(
            entity.start,
            entity.end,
            word.start,
            word.end,
        ):

            matched.append(word)

    return matched

In [22]:
def find_redactions(
    pages: List[PageText],
):

    redactions = []

    for page in pages:

        if not page.text.strip():
            continue

        results = detect_pii(
            page.text
        )

        print(
            f"\nPage {page.page_number}: "
            f"{len(results)} PII entities"
        )

        for result in results:

            matched_words = entity_to_words(
                result,
                page.words,
            )

            rect = words_to_rect(
                matched_words
            )

            if rect is None:
                continue

            value = page.text[
                result.start:result.end
            ]

            # DO NOT log sensitive values in
            # real production code.

            print(
                f"  {result.entity_type}: "
                f"{value!r} "
                f"score={result.score:.2f}"
            )

            redactions.append({
                "page": page.page_number,
                "rect": rect,
                "entity": result.entity_type,
                "score": result.score,
            })

    return redactions

In [23]:
def create_masked_pdf(
    input_pdf: str,
    output_pdf: str,
    redactions,
):

    doc = fitz.open(input_pdf)

    for item in redactions:

        page_number = item["page"]

        page = doc[
            page_number - 1
        ]

        rect = item["rect"]

        page.add_redact_annot(
            rect,
            fill=(0, 0, 0),
        )

    # Actually remove underlying content.
    for page in doc:

        page.apply_redactions(
            images=fitz.PDF_REDACT_IMAGE_PIXELS,
            graphics=(
                fitz.PDF_REDACT_LINE_ART_REMOVE_IF_COVERED
            ),
            text=fitz.PDF_REDACT_TEXT_REMOVE,
        )

    doc.save(
        output_pdf,
        garbage=4,
        deflate=True,
        clean=True,
    )

    doc.close()

    return output_pdf

In [24]:
pages = extract_pages(
    pdf_name
)

print(
    f"Loaded {len(pages)} pages"
)

Loaded 3 pages


In [27]:
redactions = find_redactions(
    pages
)

print(
    f"\nTotal redaction regions: "
    f"{len(redactions)}"
)


Page 1: 17 PII entities
  DATE_TIME: '03/10/2024' score=0.95
  DATE_TIME: '09/09/2022' score=0.95
  DATE_TIME: '05/08/1985' score=0.95
  DATE_TIME: '26/10/2022' score=0.95
  DATE_TIME: 'Annual' score=0.85
  PERSON: 'Whey Brid' score=0.85
  LOCATION: 'United Kingdom' score=0.85
  LOCATION: 'UK' score=0.85
  DATE_TIME: 'Last 12 months' score=0.85
  DATE_TIME: 'daily' score=0.85
  MEMBER_ID: 'dummy690233' score=0.70
  DATE_TIME: '16/12/22' score=0.60
  DATE_TIME: '16/01/23' score=0.60
  DATE_TIME: '16/04/23' score=0.60
  DATE_TIME: '16/03/24' score=0.60
  DATE_TIME: '16/04/24' score=0.60
  DATE_TIME: '16/06/24' score=0.60

Page 2: 1 PII entities
  URL: 'https://www.icaew.com/regul-business-restructuring-and-insolvency/creditors-guides.' score=0.60

Page 3: 8 PII entities
  PERSON: 'Lin Dummy' score=0.85
  DATE_TIME: '9 September 2022 to 8 September 2024' score=0.85
  DATE_TIME: '480.03 1,230.06' score=0.85
  DATE_TIME: '09/09/23' score=0.60
  DATE_TIME: '08/09/24' score=0.60
  DATE_TIME:

In [28]:
def words_to_rect(words):

    if not words:
        return None

    x0 = min(w.x0 for w in words)
    y0 = min(w.y0 for w in words)

    x1 = max(w.x1 for w in words)
    y1 = max(w.y1 for w in words)

    # Small padding
    padding_x = 1.5
    padding_y = 1.0

    return fitz.Rect(
        x0 - padding_x,
        y0 - padding_y,
        x1 + padding_x,
        y1 + padding_y,
    )

In [29]:
output_pdf = "masked.pdf"

create_masked_pdf(
    input_pdf=pdf_name,
    output_pdf=output_pdf,
    redactions=redactions,
)

print(
    f"Created: {output_pdf}"
)

Created: masked.pdf


In [30]:
def extract_all_text(pdf_path):

    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        text += page.get_text()

    doc.close()

    return text

In [31]:
original_text = extract_all_text(
    pdf_name
)

masked_text = extract_all_text(
    output_pdf
)

print(
    "Original characters:",
    len(original_text)
)

print(
    "Masked characters:",
    len(masked_text)
)

Original characters: 3380
Masked characters: 2969


In [32]:
detected_values = []

for page in pages:

    results = detect_pii(
        page.text
    )

    for result in results:

        value = page.text[
            result.start:result.end
        ]

        detected_values.append(
            value
        )

In [33]:
remaining = []

for value in detected_values:

    if value.casefold() in masked_text.casefold():

        remaining.append(value)

In [34]:
if remaining:

    print(
        "❌ MASKING VERIFICATION FAILED"
    )

    print(
        "Potentially remaining values:"
    )

    for value in remaining:
        print(repr(value))

else:

    print(
        "✅ MASKING VERIFICATION PASSED"
    )

✅ MASKING VERIFICATION PASSED


In [35]:
files.download(
    "masked.pdf"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>